# Testar Modeller

## Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.validate_process import preprocess_data

## Utvärderingsfunktion

In [2]:
def evaluate_model(model_name, y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)

    print(f"\n=== {model_name} ===")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }

## Läs Data

In [3]:
df = pd.read_csv(
    "../data/merged/train_merged_2023-05-26_2025-12-31_pm25tidsserie_6247_vaderstationid_52350.csv"
)

print(df.shape)

(22825, 7)


In [4]:
print(df["tid"].duplicated().sum())

0


## Preprocessing

In [5]:
df = preprocess_data(df)

print(df.shape)
df.isna().sum()

(22825, 7)


tid                         0
pm25                      144
Lufttemperatur            531
Vindriktning              568
Vindhastighet             568
Relativ luftfuktighet     568
Nederbördsmängd          1037
dtype: int64

## Dataset för Modeller

In [6]:
df["target"] = df["pm25"].shift(-6)

In [7]:
features = [
    "pm25",
    "Lufttemperatur",
    "Vindriktning",
    "Vindhastighet",
    "Relativ luftfuktighet",
    "Nederbördsmängd",
]

In [8]:
model_df = df.dropna(
    subset=features + ["target"]
)

print(model_df.shape)

model_df.isna().sum()

(21609, 8)


tid                      0
pm25                     0
Lufttemperatur           0
Vindriktning             0
Vindhastighet            0
Relativ luftfuktighet    0
Nederbördsmängd          0
target                   0
dtype: int64

## Modell: Baseline

In [ ]:
results = []   # Resultat sparas här

In [10]:
y_true = model_df["target"]
y_pred = model_df["pm25"]

baseline_results = evaluate_model(
    "Baseline",
    y_true,
    y_pred
)

results.append(baseline_results)


=== Baseline ===
MAE:  2.621
RMSE: 4.178
R²:   0.564


## Modell 1: Linerar Regression

In [11]:
X = model_df[features]
y = model_df["target"]

lr = LinearRegression()

lr.fit(X, y)

pred_lr = lr.predict(X)

lr_results = evaluate_model(
    "Linear Regression",
    y,
    pred_lr
)

results.append(lr_results)


=== Linear Regression ===
MAE:  2.527
RMSE: 3.941
R²:   0.612


## Model 2: Random Forrest

Börjar litet så det går snabbt att köra

In [12]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

pred_rf = rf.predict(X)

rf_results = evaluate_model(
    "Random Forest",
    y,
    pred_rf
)

results.append(rf_results)


=== Random Forest ===
MAE:  0.849
RMSE: 1.322
R²:   0.956


## Modell 3: HistGradientBoosting

In [13]:
hgb = HistGradientBoostingRegressor(
    random_state=42
)

hgb.fit(X, y)

pred_hgb = hgb.predict(X)

hgb_results = evaluate_model(
    "HistGradientBoosting",
    y,
    pred_hgb
)

results.append(hgb_results)


=== HistGradientBoosting ===
MAE:  2.143
RMSE: 3.184
R²:   0.747


## Jämförelse

In [14]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    "RMSE",
    ascending=True
)

,Model,MAE,RMSE,R2
2,Random Forest,0.848897,1.321644,0.956406
3,HistGradientBoosting,2.143390,3.184169,0.746959
1,Linear Regression,2.527372,3.941136,0.612348
0,Baseline,2.621101,4.177564,0.564442
